# Endoscopy SRGAN Training — YIQ Experiment

Separate, opt-in experiment: trains a Y-channel (YIQ) generator instead of the
paper-faithful V-channel (HSV) one. Uses its own files (`enhance_yiq.py`,
`train_srgan_yiq.py`) and its own checkpoint dir/weights (`srgan_yiq.pth`) —
none of this touches the paper-faithful pipeline or its checkpoints.

Mirrors the clean HSV notebook's structure (local-disk data, Drive backup only).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Kaggle authentication

**Only needed the very first time ever** — skip if a Drive backup already exists (cell 4).

In [ ]:
import os

KAGGLE_TOKEN = 'PASTE_YOUR_TOKEN_HERE'

if KAGGLE_TOKEN != 'PASTE_YOUR_TOKEN_HERE':
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(KAGGLE_TOKEN)
    os.system('chmod 600 /root/.kaggle/access_token')
    print('Kaggle token saved.')
else:
    print('Skipped -- fine if a Drive backup already exists (cell 4 will use it).')

## 3. Install dependencies

In [ ]:
!pip install -q kaggle opencv-python-headless torch torchvision scikit-image lpips

## 4. Get the dataset — local disk first, Drive as backup only

Same dataset as the HSV pipeline (YIQ is just a different color-space split of
the same images) — if you already built a Drive backup from the HSV notebook,
this cell reuses it directly, no re-download needed.

In [ ]:
import os

LOCAL_DATA = '/content/data'
DRIVE_BACKUP = '/content/drive/MyDrive/endoscopy_srgan/data_backup'
DATASETS = ['kvasir', 'cvc_clinicdb', 'etis_larib']

def _has_all(root):
    return all(os.path.isdir(f'{root}/{d}') for d in DATASETS)

if _has_all(LOCAL_DATA):
    print('Local data already present this session -- skipping.')
elif _has_all(DRIVE_BACKUP):
    print('Restoring from Drive backup (fast local copy)...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    for d in DATASETS:
        os.system(f'cp -r {DRIVE_BACKUP}/{d} {LOCAL_DATA}/')
    print('Restored from Drive backup.')
else:
    print('No local data or Drive backup found -- downloading fresh from Kaggle to local disk...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    os.system(f'kaggle datasets download -d meetnagadia/kvasir-dataset -p {LOCAL_DATA}/kvasir --unzip')
    os.system(f'kaggle datasets download -d balraj98/cvcclinicdb -p {LOCAL_DATA}/cvc_clinicdb --unzip')
    os.system(f'kaggle datasets download -d nguyenvoquocduong/etis-laribpolypdb -p {LOCAL_DATA}/etis_larib --unzip')
    print('Downloaded. Backing up to Drive for future sessions...')
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    for d in DATASETS:
        os.system(f'cp -r {LOCAL_DATA}/{d} {DRIVE_BACKUP}/')
    print('Backed up to Drive.')

## 5. Build the manifest (local paths)

In [ ]:
import glob, random, json

paths = []
paths += glob.glob(f'{LOCAL_DATA}/kvasir/kvasir-dataset/*/*.jpg')
paths += glob.glob(f'{LOCAL_DATA}/cvc_clinicdb/PNG/Original/*.png')
paths += glob.glob(f'{LOCAL_DATA}/etis_larib/images/*.png')
print('total images found:', len(paths))

random.seed(42)
random.shuffle(paths)
n_val = int(0.1 * len(paths))
manifest = {'train': paths[n_val:], 'val': paths[:n_val]}

MANIFEST_PATH = f'{LOCAL_DATA}/manifest.json'
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f)
print('train:', len(manifest['train']), '| val:', len(manifest['val']))

## 6. Clone or update the repo

In [ ]:
import os

if os.path.exists('/content/repo/.git'):
    %cd /content/repo
    !git pull
else:
    !git clone https://github.com/knah1d/unsharp-image_processing.git /content/repo
    %cd /content/repo

## 7. Smoke test (recommended before a long run)

Cheap sanity check on the YIQ pipeline specifically -- catches bugs in ~1 minute.

In [ ]:
!python train_srgan_yiq.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_yiq_smoketest \
    --epochs 3 --pretrain_epochs 1 --batch_size 8

## 8. Real training run

Checkpoints go to a YIQ-specific Drive folder (`checkpoints_yiq`), fully
separate from the HSV pipeline's `checkpoints`. Auto-resumes via
`srgan_yiq_last.pth` if this cell is rerun after a disconnect.

In [ ]:
!python train_srgan_yiq.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_yiq \
    --epochs 50 --pretrain_epochs 5 --batch_size 16

## 9. (Optional) Warm-restart if it plateaus

Same fix as the HSV pipeline if training stalls: bumps LR back up and runs a
fresh cosine decay over the epochs remaining, rather than continuing an
exhausted schedule.

In [ ]:
!python train_srgan_yiq.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_yiq \
    --epochs 80 --fresh_schedule